# QC-C1-01: Circuit Profiling and Limited Basis-Gate Decomposition
PESU – C-DAC Quantum Hackathon 2026

**How to run in Qniverse JupyterLab:** upload the repository folder (or `git clone` it) next to this notebook so that `src/qnvkit/` exists, then run all cells. Only `numpy` and the Qniverse QSDK are needed.

The first cell reports whether the **official Qniverse QSDK** is in use. Outside Qniverse a local stand-in with the same documented calls is used instead.

In [ ]:
import os, sys
for candidate in ("src", "../src", "CDAC-Hack/src"):
    if os.path.isdir(os.path.join(candidate, "qnvkit")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

import qnvkit.qsdk_adapter as adapter          # must be imported before Qniverse
from qnvkit.metrics import profile
from qnvkit.decompose.decomposer import decompose

from Qniverse.QniverseCircuit import *
from Qniverse.gates import *
import numpy as np

print("QSDK in use:", adapter.qsdk_description())

## 1. Build a circuit with the QSDK

In [ ]:
qc = QniverseCircuit()
q = QuantumRegister(3, 'q')
c = ClassicalRegister(3, 'c')
qc.add_register(q)
qc.add_register(c)

qc.add_gate(h, q[0])
qc.add_gate(s, q[1])
qc.add_gate(t, q[2])
qc.add_gate(cx, q[0], q[1])
qc.add_gate(cz, q[1], q[2])
qc.add_gate(swap, q[0], q[2])
qc.add_gate(crz(0.7), q[0], q[1])
qc.add_gate(ccx, q[0], q[1], q[2])
qc.add_gate(barrier, q[0], q[1], q[2])
for i in range(3):
    qc.add_gate(measure, q[i], c[i])

original_qasm = adapter.read_qasm(qc)
print(original_qasm)

## 2. Profile
Depth uses an earliest-available-layer schedule: each operation takes one layer on every qubit it uses, operations sharing a qubit keep circuit order, operations on disjoint qubits share a layer. Measure and reset take a layer; a barrier takes none but aligns its qubits.

In [ ]:
report = profile(qc)
print(report.summary())
report.to_dict()

## 3. Decompose into a chosen basis
Returns a **new** `QniverseCircuit`; metrics before and after; equivalence checked by unitary comparison up to global phase.

In [ ]:
res = decompose(qc, {"rx", "ry", "rz", "cx"})
print(res.summary())
print(adapter.read_qasm(res.circuit))

In [ ]:
for basis in ({"u3", "cx"}, {"rz", "ry", "cz"}, {"rx", "rz", "cx"}, {"p", "ry", "cx"}):
    r = decompose(qc, basis)
    print(sorted(basis), "| gates", r.before.total_gates, "->", r.after.total_gates,
          "| depth", r.before.depth, "->", r.after.depth, "| verified:", r.verified)

## 4. Unsupported operations are reported, never emitted

In [ ]:
for basis in ({"rx", "ry", "rz"}, {"h", "t", "cx"}):
    r = decompose(qc, basis)
    print("Basis", sorted(basis), "-> circuit:", r.circuit)
    print(r.unsupported.summary(), "\n")

for bad in (set(), {"cx", "sx"}):
    try:
        decompose(qc, bad)
    except ValueError as e:
        print("Rejected", sorted(bad), ":", str(e)[:100])

## 5. Input circuit is unchanged

In [ ]:
print("Input QASM identical before/after:", adapter.read_qasm(qc) == original_qasm)
print("Output is a different object:", res.circuit is not qc)

## 6. Run original and decomposed circuits on a Qniverse simulator
Paste your Qniverse API key below (2 CPU jobs = 10 credits). The two histograms should agree within sampling noise.

In [ ]:
token = ""   # <- your Qniverse API key (keep it out of git)

if token.strip():
    from Qniverse.backend import Backend
    from Qniverse.api import run, fetch_job_result, job_status
    import time

    backend = Backend()
    backend.processor("CPU")
    backend.hpc("QACC_Cluster")
    backend.simulator("qasm_simulator")

    for label, circ in (("original", qc), ("decomposed", res.circuit)):
        job = run(circ, backend, 1024, token=token, job_name="c1_" + label)
        while "COMPLETED" not in str(job_status(token, job)).upper():
            time.sleep(3)
        print(label, fetch_job_result(token, job))
else:
    print("No token set: skipped.")